# FIN 4600 Lab: How a Smart Contract Really Works

**Fintech Foundations · Michigan Technological University · goes with study guide gen-08 (Ethereum)**

In this lab you will run real Ethereum smart contracts on a **private blockchain that lives inside this notebook**. All the code is written for you. Your job is to **run it, read it, predict what will happen, try small experiments, and explain what you saw.** You do not need to know how to program.

**The main lesson is how cryptocurrency and smart contracts work.** Along the way, the comments in the code (the lines starting with `#`) explain what each line of Python does, so you will also pick up how Python and Jupyter notebooks work.

| Mission | What you will see | Points |
|---|---|---|
| 1 | Accounts, addresses, and why ether is counted in wei | 10 |
| 2 | Your first contract: a free *read* vs. a paid *transaction*, and what gas costs | 15 |
| 3 | Access control: one line of code decides who is allowed to act | 15 |
| 4 | The escrow from Nofsinger Figure 5.3, with no bank in the middle | 15 |
| 5 | A token the issuer can freeze, like USDT and USDC | 10 |
| 6 | The 2016 DAO hack, re-created, and the two-line fix | 20 |
| 7 | What it all cost, and what a Layer 2 changes | 15 |

**Nothing here touches a real network.** There is no wallet, no seed phrase and no money. The blockchain disappears when you close the notebook.

### Symbols used in this notebook

* 🔮 **Predict:** write down what you expect *before* you run the next cell.
* 🧪 **Experiment:** change one value in the code where it says so, run the cell again, and record what happened.
* ✍️ **Answer:** a written question that is graded. Double-click the cell to type in it, then press Shift+Enter.
* ✅ / ⛔ / ❌ in the output: something worked / the contract refused a request / a check did not pass.

## How to use a Jupyter notebook (read this first)

* A notebook is a list of **cells**. This cell is a *text* (Markdown) cell. Cells with a grey background and `[ ]:` on the left are *code* cells.
* Click a code cell and press **Shift+Enter** to run it. The output appears underneath and the cursor moves to the next cell.
* While a cell runs, the left side shows `[*]`. When it finishes, it shows a number such as `[4]`: the order in which cells were run.
* Everything you create (variables, the blockchain itself) lives in the notebook's memory, called the **kernel**. Run cells **from top to bottom**, because later cells use things made by earlier ones.
* If something gets confusing: menu **Kernel → Restart Kernel and Run All Cells**. That wipes memory and runs everything again in order.
* If a code cell's last line is just a name (like `costs`), Jupyter displays it as a table or value.

### Warm-up: Python in one cell

Run the cell below and compare each line with its output. Every idea here is used later in the lab.

In [ ]:
# A line that starts with # is a COMMENT. Python ignores it; it is a note for people.

# A VARIABLE is a name that holds a value. "=" means "store the value on the right under the name on the left".
course = "FIN 4600"            # text goes in quotes and is called a "string"
price_of_ether = 2_330         # a number; the underscore is just a readable thousands separator

# print(...) is a FUNCTION: a named command. The things in the parentheses are its inputs ("arguments").
print(course)

# An f-string (note the f before the quote) fills in variables inside {curly braces}.
print(f"One ether costs about ${price_of_ether} in this lab.")

# A LIST holds several values in order, inside square brackets.
names = ["Alice", "Bob", "Carol"]
print(names[0])                # positions start at 0, so names[0] is the first item: Alice

# A DICTIONARY looks values up by a key, like a phone book: {key: value}.
balances = {"Alice": 10, "Bob": 3}
print(balances["Bob"])         # look up Bob's value: 3

# A FOR LOOP repeats the indented lines once for each item in the list.
for person in names:
    print("Hello,", person)

## Mission 0: Start the blockchain

The next cell loads the lab's tools and starts a fresh private Ethereum blockchain. The first time you run it, it downloads the Solidity **compiler** (about 10 MB). A compiler translates human-readable contract code into the bytecode that the Ethereum Virtual Machine (EVM) runs.

The tools come from `helpers/chain.py` in this folder. You never have to open that file, but it is commented too if you are curious.

In [ ]:
# "from X import a, b, c" loads specific tools (functions) from the file helpers/chain.py.
# The parentheses just let the list run over several lines.
from helpers.chain import (
    setup_compiler,   # downloads and selects the Solidity compiler
    read_contract,    # reads a contract's source code from the contracts/ folder
    compile_source,   # translates Solidity source code into EVM bytecode
    new_chain,        # starts a brand-new private blockchain in memory
    deploy,           # puts a compiled contract onto the blockchain
    transact,         # sends a transaction (it costs gas and can change data)
    try_transact,     # same, but if the contract says "no", prints why instead of stopping
    balance_eth,      # looks up how much ether an account or contract holds
    time_travel,      # moves the test blockchain's clock forward
    events,           # lists the log messages ("events") a contract has written
    gas_table,        # a table of every transaction we sent and the gas each one used
    check,            # prints ✅ if a statement is true, ❌ if not
)

# pandas is Python's most popular table library. "as pd" gives it a short nickname.
import pandas as pd

# Call the setup function. Its result is a short message, which print() displays.
print(setup_compiler())

# Start the blockchain. The result is an object (a bundle of data plus tools) that we store
# in a variable called w3, short for "Web3", the name of the Python library that talks to Ethereum.
w3 = new_chain()

# Objects have their own functions, called METHODS, reached with a dot: w3.is_connected()
print("Connected to our private test blockchain:", w3.is_connected())

## Mission 1: Accounts and ether (10 points)

On Ethereum an **account** is identified by an **address**: a 20-byte number written in hexadecimal (digits 0–9 and letters a–f), starting with `0x`. Whoever holds the account's **private key** can sign transactions for it. Our test blockchain creates 10 accounts and gives each one ether to play with.

Ether is divisible into tiny units called **wei**: 1 ether = 1,000,000,000,000,000,000 wei (10¹⁸). Gas prices are quoted in **gwei** (a billion wei).

In [ ]:
# w3.eth.accounts is a LIST of the 10 test account addresses.
accounts = w3.eth.accounts

# "Unpacking": the first four addresses go into four variables at once, in order.
# accounts[:4] means "items 0, 1, 2 and 3 of the list".
alice, bob, carol, dave = accounts[:4]

# accounts[9] is the tenth (last) account. Mallory will be our attacker in Mission 6.
mallory = accounts[9]

# A dictionary that turns an address into a friendly name, so outputs are easier to read.
names = {alice: "Alice", bob: "Bob", carol: "Carol", dave: "Dave", mallory: "Mallory"}

# Build a table: one row per account. The part in [ ... for a in accounts ] is a "list comprehension":
# it runs the dictionary {...} once for every address a in the list, collecting the rows.
# names.get(a, "-") looks up a's name and uses "-" if it has none.
table = pd.DataFrame([
    {"name": names.get(a, "-"), "address": a, "balance_eth": balance_eth(w3, a)}
    for a in accounts
])

table   # a bare name on the last line: Jupyter displays it as a table

In [ ]:
# How big is one ether in wei? w3.to_wei converts a number of ether into wei.
one_ether_in_wei = w3.to_wei(1, "ether")
print(f"1 ether = {one_ether_in_wei:,} wei")          # the ",": print with thousands separators

# And the other way: w3.from_wei converts wei back into ether (or gwei).
print("2 gwei =", w3.from_wei(w3.to_wei(2, "gwei"), "ether"), "ether")

# Balances are stored on the chain in wei. Here is Alice's raw balance, exactly as Ethereum keeps it.
raw = w3.eth.get_balance(alice)
print(f"Alice's balance in wei:   {raw:,}")
print(f"Alice's balance in ether: {balance_eth(w3, alice):,}")

✍️ **1a (5 pts).** An Ethereum address is public, like an account number, but only the holder of the matching private key can move its ether. Why is it safe to share your address but never your private key?

*Your answer:*

✍️ **1b (5 pts).** Ethereum keeps every balance as a whole number of wei instead of a decimal number of ether. Why might a financial system prefer whole numbers? (Hint: think about rounding.)

*Your answer:*

## Mission 2: Your first contract (15 points)

Here is `SimpleStorage`, the first example from the lecture. It stores one number, `storedRate`. Solidity uses `//` for comments, where Python uses `#`.

In [ ]:
# read_contract reads the text of a file in the contracts/ folder and returns it as a string.
storage_src = read_contract("SimpleStorage.sol")
print(storage_src)

**Compiling** turns the source into **bytecode** (what the EVM runs) and an **ABI** (Application Binary Interface): a menu of functions that outside programs, like this notebook, are allowed to call.

**Deploying** is itself a transaction: it stores the bytecode on the blockchain at a new address. From then on, every node holds a copy and anyone can call it.

In [ ]:
# Compile. The result is a dictionary holding the ABI and the bytecode.
storage_compiled = compile_source(storage_src, "SimpleStorage")

# Pull the function names out of the ABI (only the entries whose "type" is "function").
menu = [item["name"] for item in storage_compiled["abi"] if item["type"] == "function"]
print("Functions on the menu (the ABI):", menu)

# Bytecode is stored as hexadecimal text: 2 characters per byte, so divide the length by 2.
# "//" in Python means whole-number division.
print("Bytecode size:", len(storage_compiled["bytecode"]) // 2, "bytes")
print("First bytes of the bytecode:", storage_compiled["bytecode"][:40], "...")

# Deploy the contract, sending the transaction from Alice's account.
# sender=alice is a KEYWORD ARGUMENT: the input is named so its meaning is clear.
storage = deploy(w3, storage_compiled, sender=alice)

### A free *call* vs. a paid *transaction*

* A **call** only *reads*. Any node can answer it from its own copy of the chain, so it is free and changes nothing.
* A **transaction** *writes*. It must be signed, put into a block, and run by every node, so it costs **gas**.

🔮 **Predict:** the next cell reads the rate, changes it, and reads it again. Which of those three steps will show a gas cost?

In [ ]:
# 🧪 EXPERIMENT (Mission 2): change this number, then run this cell again.
NEW_RATE = 525

# storage.functions.storedRate() picks the storedRate function off the contract's menu;
# .call() asks for its value. A call is free and creates no transaction.
print("Rate before:", storage.functions.storedRate().call())

# transact(...) sends a real transaction from Alice that runs setRate(NEW_RATE) on every node.
# It returns a RECEIPT: the blockchain's record of what happened (block number, gas used, ...).
receipt = transact(w3, storage.functions.setRate(NEW_RATE), sender=alice)

print("Rate after: ", storage.functions.storedRate().call())
print("Recorded in block number:", receipt.blockNumber)

# The contract also wrote an EVENT (a log message). Show all RateChanged events as a table.
events(storage, "RateChanged")

### What did that cost?

On the real Ethereum network you pay **gas used × gas price**. We use an illustrative gas price of **2 gwei** and an ether price of **$2,330** (November 4, 2026).

In [ ]:
# Constants are often written in CAPITALS by convention. We reuse these in Mission 7.
GAS_PRICE_GWEI = 2        # price of one unit of gas, in gwei
ETH_USD = 2330            # dollars per ether

gas_used = receipt.gasUsed                       # how much gas setRate used (read from the receipt)
fee_eth = gas_used * GAS_PRICE_GWEI * 1e-9       # 1e-9 is scientific notation for 0.000000001 (1 gwei in ether)
fee_usd = fee_eth * ETH_USD                      # convert ether to dollars

# Inside {...}, the part after ":" formats the number: ",": thousands separator; ".10f": 10 decimals.
print(f"Gas used: {gas_used:,}")
print(f"Fee: {gas_used:,} gas × {GAS_PRICE_GWEI} gwei = {fee_eth:.10f} ETH ≈ ${fee_usd:.3f}")

🔮 **Predict:** Dave did not deploy this contract. If Dave tries to change the rate, will the contract let him?

In [ ]:
# Dave (not the deployer) sends a transaction asking to set the rate to 9999.
transact(w3, storage.functions.setRate(9999), sender=dave)

# Read the rate back. Did Dave's change stick?
print("Rate now:", storage.functions.storedRate().call())

# check(statement, message if true, message if false) prints ✅ or ❌.
check(storage.functions.storedRate().call() == 9999,
      "Dave was able to change the rate", "Dave could not change the rate")

✍️ **2a (5 pts).** Your 🔮 prediction about which step costs gas: were you right? In your own words, why is reading free but writing paid?

*Your answer:*

✍️ **2b (4 pts).** 🧪 Change `NEW_RATE` to a different number and re-run that cell. Did the gas used change much? Did the fee change much? (Record the numbers.)

*Your answer:*

✍️ **2c (6 pts).** Dave is a stranger, yet he changed the rate. Why? If this number were the interest rate on a loan contract, what could go wrong?

*Your answer:*

## Mission 3: One line decides who may act (15 points)

`OwnedStorage` fixes Dave's trick with two additions:

* a **constructor**, which runs once when the contract is deployed and records the deployer as `owner`, and
* a **`require`** line: if its condition is false, the contract refuses ("reverts") and **nothing changes**.

In Solidity, `msg.sender` always means "the account that sent this transaction". Nobody can fake it, because every transaction is signed with the sender's private key.

In [ ]:
# A Python string in triple quotes """ ... """ can span many lines. We keep the Solidity code in one,
# so you can edit it right here for the experiment below.
owned_src = """
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.24;

contract OwnedStorage {
    uint256 public storedRate;
    address public owner;

    constructor() {
        owner = msg.sender;            // whoever deploys the contract becomes the owner
    }

    function setRate(uint256 newRate) external {
        require(msg.sender == owner, "only owner");   // the one line that decides who may act
        storedRate = newRate;
    }
}
"""

# Compile and deploy in one line: compile_source(...) runs first, and its result goes straight into deploy(...).
owned = deploy(w3, compile_source(owned_src, "OwnedStorage"), sender=alice)
print("Owner recorded by the constructor:", names[owned.functions.owner().call()])

🔮 **Predict:** Alice tries to set the rate to 450, then Dave tries to set it to 1. What happens to each?

In [ ]:
# try_transact returns TWO values (a "tuple"): whether it worked (True/False), and the receipt or the refusal reason.
# We unpack them into two variables each.
ok_alice, _ = try_transact(w3, owned.functions.setRate(450), sender=alice, label="Alice sets 450")   # "_" = a value we ignore
ok_dave, reason = try_transact(w3, owned.functions.setRate(1), sender=dave, label="Dave tries 1")

print("Rate now:", owned.functions.storedRate().call())
check(ok_alice, "the owner can set the rate", "the owner could not set the rate")
check(not ok_dave, "the stranger was refused: " + str(reason), "the stranger was NOT refused")

🧪 **Experiment:** in the code cell that defines `owned_src` (two cells up), put `//` at the very start of the `require(...)` line so Solidity treats it as a comment. Run that cell, then run the Alice-and-Dave cell again. **Record what happens.** Then remove the `//` and run both cells again to restore the protection.

✍️ **3a (8 pts).** What happened in the 🧪 experiment when the `require` line was commented out? What does that tell you about how much of a contract's security can rest on a single line of code?

*Your answer:*

✍️ **3b (7 pts).** Dave's refused transaction still appeared in the output with a reason. On real Ethereum, a refused (reverted) transaction is still recorded in a block and the sender still pays for the gas used up to the refusal. Why do you think the network charges for a failed transaction?

*Your answer:*

## Mission 4: The escrow from Figure 5.3 (15 points)

Alice is buying a used bike from Bob for **2 ether**. Instead of a bank in the middle, Alice deploys an escrow contract and deposits the money into it. Only Alice can release the money to Bob. If Bob never delivers, Alice can take a refund after the deadline. Read the contract: every `require` is one of the deal's conditions.

In [ ]:
escrow_src = read_contract("Escrow.sol")
print(escrow_src)
escrow_compiled = compile_source(escrow_src, "Escrow")

In [ ]:
# Remember Bob's balance now, so we can see what he gains later.
bob_start = balance_eth(w3, bob)

# Deploy the escrow. The extra inputs bob and 7 go to the contract's constructor: (seller, daysToDeliver).
# value_eth=2 sends 2 ether along with the deployment: that is Alice's deposit.
escrow = deploy(w3, escrow_compiled, bob, 7, sender=alice, value_eth=2,
                label="deploy Escrow (Alice deposits 2 ETH)")

print("The escrow contract now holds:", balance_eth(w3, escrow.address), "ETH")
print("Buyer: ", names[escrow.functions.buyer().call()])
print("Seller:", names[escrow.functions.seller().call()])

🔮 **Predict:** before running the next cell, write *succeeds* or *refused* (and why) for each of these four attempts, in order:

1. Bob tries to release the money to himself.
2. Alice asks for a refund on day 0.
3. Alice confirms delivery.
4. Alice confirms delivery a second time.

*Your predictions:* 1. ___ 2. ___ 3. ___ 4. ___

In [ ]:
# Four attempts, in order. try_transact prints ✅ if the contract accepted, ⛔ and the reason if it refused.
try_transact(w3, escrow.functions.confirmDelivery(), sender=bob,   label="1. Bob tries to pay himself")
try_transact(w3, escrow.functions.refund(),          sender=alice, label="2. Alice asks for a refund on day 0")
try_transact(w3, escrow.functions.confirmDelivery(), sender=alice, label="3. Alice confirms delivery")
try_transact(w3, escrow.functions.confirmDelivery(), sender=alice, label="4. Alice confirms again")

# round(x, 6) rounds to 6 decimal places.
print("Bob gained:", round(balance_eth(w3, bob) - bob_start, 6), "ETH")
print("Escrow now holds:", balance_eth(w3, escrow.address), "ETH")

Now the unhappy path: Carol buys a bike from Dave for 1 ether, and Dave never delivers. Our test blockchain lets us move its clock forward, something you obviously cannot do on the real network.

In [ ]:
# 🧪 EXPERIMENT (Mission 4): change DAYS_LATER to 5 and run this cell again. What changes?
DAYS_LATER = 8

# A second escrow: Carol is the buyer, Dave the seller, 1 ether deposit, 7 days to deliver.
escrow2 = deploy(w3, escrow_compiled, dave, 7, sender=carol, value_eth=1,
                 label="deploy Escrow (Carol deposits 1 ETH)")

# Move the blockchain's clock forward, then Carol asks for her money back.
time_travel(w3, DAYS_LATER)
try_transact(w3, escrow2.functions.refund(), sender=carol, label=f"Carol asks for a refund on day {DAYS_LATER}")

print("Escrow 2 now holds:", balance_eth(w3, escrow2.address), "ETH")

✍️ **4a (5 pts).** Compare your four 🔮 predictions with what happened. For each one you got wrong (if any), which `require` line in the contract explains the result?

*Your answer:*

✍️ **4b (4 pts).** 🧪 What happened to Carol's refund with `DAYS_LATER = 5`, and why?

*Your answer:*

✍️ **4c (6 pts).** The escrow replaced the banks in Figure 5.3. Name one job the banks did that this contract cannot do, and who or what would have to do it instead.

*Your answer:*

## Mission 5: A token the issuer can freeze (10 points)

Alice issues 1,000,000 **Husky Dollars**. A token is not a coin that moves around: it is a **ledger inside a contract**, a `mapping` (Solidity's version of a dictionary) from each address to its balance. A "transfer" just subtracts from one entry and adds to another. Notice the `freeze` function at the bottom.

In [ ]:
token_src = read_contract("ClassToken.sol")
print(token_src)

# Deploy with 1,000,000 Husky Dollars, all credited to Alice (the issuer) by the constructor.
token = deploy(w3, compile_source(token_src, "ClassToken"), 1_000_000, sender=alice)

# Alice pays Carol 500 Husky Dollars.
transact(w3, token.functions.transfer(carol, 500), sender=alice, label="Alice pays Carol 500")

In [ ]:
# A small FUNCTION we define ourselves with "def". It prints everyone's token balance.
# The indented lines are the function's body; they run each time we call show_balances().
def show_balances():
    for person in (alice, carol, dave):            # loop over three addresses
        amount = token.functions.balanceOf(person).call()
        frozen = token.functions.frozen(person).call()
        print(f"{names[person]:6} {amount:>9,} Husky Dollars   frozen: {frozen}")

show_balances()

🔮 **Predict:** Alice freezes Carol's account. Then Carol tries to pay Dave 100, and Dave tries to freeze Alice. What happens to each?

In [ ]:
transact(w3, token.functions.freeze(carol), sender=alice, label="Alice (issuer) freezes Carol")
try_transact(w3, token.functions.transfer(dave, 100), sender=carol, label="Carol tries to pay Dave 100")
try_transact(w3, token.functions.freeze(alice), sender=dave, label="Dave tries to freeze Alice")

show_balances()

✍️ **5a (5 pts).** Who could move Carol's 500 Husky Dollars after the freeze? Is this token really "decentralized"? Explain.

*Your answer:*

✍️ **5b (5 pts).** Tether (USDT) and Circle (USDC) have freeze (blacklist) functions like this one, and used them after the October 2026 Binance theft (ce-02). Give one argument that this power is good for users and one that it is a risk.

*Your answer:*

## Mission 6: Re-create the DAO hack, then fix it (20 points)

In June 2016 an attacker drained about 3.6 million ether from The DAO. The bug was in the **order of two steps** in a withdraw function. `VulnerableBank` has the same bug: its `withdraw()` **sends the money first** and **updates its books second**.

The `Attacker` contract exploits this. When a contract receives ether, a special function called `receive()` runs automatically. The attacker's `receive()` calls `withdraw()` **again**, before the bank has set the attacker's balance to zero, so the bank still thinks the attacker is owed money.

In [ ]:
bank_src = read_contract("VulnerableBank.sol")
print(bank_src)
print(read_contract("Attacker.sol"))
attacker_compiled = compile_source(read_contract("Attacker.sol"), "Attacker")

🔮 **Predict:** three honest savers each deposit 3 ETH (the bank holds 9). Mallory deposits **1 ETH** and attacks. How much will she walk away with?

In [ ]:
# 🧪 EXPERIMENT (Mission 6): change SAVER_DEPOSIT to 5 and run this cell again.
SAVER_DEPOSIT = 3

# Start a FRESH blockchain for a clean experiment (everything from Missions 1-5 is wiped).
w3 = new_chain()
alice, bob, carol, dave = w3.eth.accounts[:4]
mallory = w3.eth.accounts[9]

# Alice opens the bank; Bob, Carol and Dave each deposit.
bank = deploy(w3, compile_source(bank_src, "VulnerableBank"), sender=alice)
for saver in (bob, carol, dave):
    transact(w3, bank.functions.deposit(), sender=saver, value_eth=SAVER_DEPOSIT,
             label=f"deposit {SAVER_DEPOSIT} ETH")
print("Bank holds:", balance_eth(w3, bank.address), "ETH of honest savers' money\n")   # "\n" = blank line

# Mallory deploys her attack contract, pointing it at the bank, and attacks with 1 ether.
attacker = deploy(w3, attacker_compiled, bank.address, sender=mallory)
transact(w3, attacker.functions.attack(), sender=mallory, value_eth=1, label="Mallory attacks with 1 ETH")

print("\nBank holds now:  ", balance_eth(w3, bank.address), "ETH")
print("Attacker holds:  ", balance_eth(w3, attacker.address), "ETH")
print("Times re-entered:", attacker.functions.reentries().call())

### The fix: change the order

Below is `SafeBank`. It is identical except that `withdraw()` sets the balance to zero **before** sending the money. This rule is called **checks, effects, interactions**: check the conditions, update your own records, and only then send money to anyone else. (The escrow in Mission 4 does the same thing: `settled = true;` comes before the payment.)

🔮 **Predict:** Mallory runs exactly the same attack against `SafeBank`. What happens?

In [ ]:
safe_src = """
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.24;

contract SafeBank {
    mapping(address => uint256) public balanceOf;

    function deposit() external payable {
        balanceOf[msg.sender] += msg.value;
    }

    function withdraw() external {
        uint256 amount = balanceOf[msg.sender];
        require(amount > 0, "nothing to withdraw");
        balanceOf[msg.sender] = 0;                         // EFFECT first: update the books
        (bool ok, ) = msg.sender.call{value: amount}("");  // INTERACTION last: send the money
        require(ok, "send failed");
    }
}
"""

w3 = new_chain()
alice, bob, carol, dave = w3.eth.accounts[:4]
mallory = w3.eth.accounts[9]

safe = deploy(w3, compile_source(safe_src, "SafeBank"), sender=alice)
for saver in (bob, carol, dave):
    transact(w3, safe.functions.deposit(), sender=saver, value_eth=3, label="deposit 3 ETH")

attacker = deploy(w3, attacker_compiled, safe.address, sender=mallory)
attacked, reason = try_transact(w3, attacker.functions.attack(), sender=mallory, value_eth=1,
                                label="Mallory attacks with 1 ETH")

# An honest saver can still withdraw normally.
try_transact(w3, safe.functions.withdraw(), sender=bob, label="Bob withdraws his 3 ETH")

check(not attacked, "the attack was refused", "the attack worked")
check(balance_eth(w3, safe.address) == 6, "Carol's and Dave's 6 ETH is safe", "the bank lost money")

✍️ **6a (6 pts).** Mallory deposited 1 ether. How much did she take out, and how many times did her contract re-enter `withdraw()`? Explain in your own words why the bank kept paying.

*Your answer:*

✍️ **6b (4 pts).** 🧪 With `SAVER_DEPOSIT = 5`, how much did Mallory take and how many re-entries were there? What does this tell you about how much of the bank is at risk?

*Your answer:*

✍️ **6c (5 pts).** Why did the same attack fail against `SafeBank`? Trace what happens when the attacker's `receive()` calls `withdraw()` the second time.

*Your answer:*

✍️ **6d (5 pts).** In 2016 the Ethereum community reversed the DAO theft with a hard fork. Could the same thing happen for a theft from a contract like this today? What does your answer say about the word "immutable"?

*Your answer:*

## Mission 7: What did it all cost? (15 points)

Every transaction since the last `new_chain()` (the `SafeBank` experiment) was logged. Let's turn gas into money.

In [ ]:
# 🧪 EXPERIMENT (Mission 7): mainnet gas prices jump on busy days. Change GAS_PRICE_GWEI to 50 and re-run.
GAS_PRICE_GWEI = 2
ETH_USD = 2330

costs = gas_table()                                              # a pandas DataFrame (a table)
costs["fee_eth"] = costs["gas_used"] * GAS_PRICE_GWEI * 1e-9     # new column: the formula runs on every row at once
costs["fee_usd"] = (costs["fee_eth"] * ETH_USD).round(3)         # new column in dollars, rounded to 3 decimals

costs   # display the table

In [ ]:
# Draw a horizontal bar chart straight from the table: one bar per operation, length = fee in dollars.
ax = costs.plot.barh(x="operation", y="fee_usd", legend=False,
                     title=f"Fee per transaction at {GAS_PRICE_GWEI} gwei and ${ETH_USD:,} per ETH")
ax.set_xlabel("US dollars")       # label the horizontal axis
ax.invert_yaxis()                 # list operations top to bottom in the order they happened

✍️ **7a (5 pts).** Which operations cost the most, and why? (Hint: what does a deployment have to store on the blockchain?) 🧪 What did the most expensive one cost at 50 gwei?

*Your answer:*

✍️ **7b (5 pts).** Using the lecture, explain how a Layer 2 rollup makes the same transaction cheaper, and what you give up.

*Your answer:*

✍️ **7c (5 pts).** Reflection: you have now run, broken and fixed smart contracts. Before you put your own money into a DeFi contract, what three things would you check?

*Your answer:*

## Submit

1. Menu **Kernel → Restart Kernel and Run All Cells**. Every cell should run; your 🧪 experiment values can be back at their originals or left changed, as long as your written answers record what you saw.
2. Make sure every ✍️ answer is filled in, then save (Ctrl+S or ⌘+S).
3. Record your work with git from the terminal (practice for the command line):

```
git add smart_contracts_lab.ipynb
git commit -m "Complete smart contracts lab"
```

4. Submit the notebook as directed in Canvas.

**Never** paste a seed phrase or private key into a notebook or website, and never deploy classroom code to the real Ethereum network.